
#  Part 1: ProtBERT and ESM-2 — From sequence to contextualised embedding

First, we get the libraries we need. For each model, we need a tokenizer and the model weights.

In [ ]:
# Import libraries
import torch
import numpy as np
from transformers import (
    # Import ProtBert tokenizer and model
    BertTokenizer, BertModel,
    # Import ESM tokenizer and model
    EsmTokenizer, EsmModel

)

# Set device to GPU  (CUDA)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Example sequences

We are going to embed really short peptides. The idea is to get a sense of how  tokenisation and embedding work in these models, so shorter sequences are better. Later on in the workshop you will work with real protein sequences.

In [ ]:

# -----------------------------------------------------------
# Sample sequences
# -----------------------------------------------------------
SEQUENCES = [
    "AAAA",
    "MKWVTFISLL",
]


#Get tokenizer and model weights for both architectures

In [ ]:
# ProtBert
protbert_tok = BertTokenizer.from_pretrained("Rostlab/prot_bert")
# It is important we set the model to eval(), which means it will embed our sequences using the frozen pretrained weights
# and not induce any stochasticity or change any weights
protbert_model = BertModel.from_pretrained("Rostlab/prot_bert").to(DEVICE).eval()


# ESM2 8M
esm2_8m_tok = EsmTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
esm2_8m_model = EsmModel.from_pretrained("facebook/esm2_t6_8M_UR50D").to(DEVICE).eval()


## Exercise 1: get a bigger ESM model
Go to https://huggingface.co/facebook/esm2_t12_35M_UR50D and choose from one of the ESM2 models to use instead of 8M.

In [ ]:

# ESM2
# esm2_bigger_tok = EsmTokenizer.from_pretrained("_______insert model__________")
# esm2_bigger_model = EsmModel.from_pretrained("____insert model_______").to(DEVICE).eval()


## Input sequences and tokenisation
1. We need to make sure we are inputting the sequences in the correct format each model expects. In this case, ProtBert needs a single space between each aminoacid, whereas ESM2 can take the sequence as continuous string that we already have.

In [ ]:
protbert_input_str = [" ".join(list(s)) for s in SEQUENCES]


2. Then, we can tokenise the input. When we decode the the tokens, we see that special tokens have been added at the start and end of the sequence.

In [ ]:
# a) Tokenise just the first sequence

###########################################################################
# PROTBERT
###########################################################################
input_protbert = protbert_tok(protbert_input_str[0], return_tensors="pt", padding=True).to(DEVICE)
print("ProtBert tokenisation of the first sequence")
print("Raw seq :", protbert_input_str[0])
print("Input to ProtBert:", input_protbert)
print("Tokens  :", protbert_tok.convert_ids_to_tokens(input_protbert["input_ids"][0]))

print(60*"=")
###########################################################################
# ESM
###########################################################################
input_esm = esm2_8m_tok(SEQUENCES[0], return_tensors="pt", padding=True).to(DEVICE)
print("ESM2 tokenisation of the first sequence")
print("Raw seq :", SEQUENCES[0])
print("Input ESM:", input_esm)
print("Tokens  :", esm2_8m_tok.convert_ids_to_tokens(input_esm["input_ids"][0]))


### Padding & the attention mask
When sequences in a batch have different lengths, we pad the shorter ones with [PAD] tokens to make them all the same length (since the model needs a rectangular batch). The attention mask keeps track of which positions are real (1) and which are just padding (0) - the model uses this to ignore the padding when computing embeddings.

In [ ]:
###########################################################################
# PROTBERT
###########################################################################
input_protbert = protbert_tok(protbert_input_str, return_tensors="pt", padding=True).to(DEVICE)
print("ProtBert tokenisation of the all sequences")
print("Raw seqs :", protbert_input_str)
print("Input ProtBert:", input_protbert)
print("Tokens for first sequence  :", protbert_tok.convert_ids_to_tokens(input_protbert["input_ids"][0]))
print("Tokens for second sequence  :", protbert_tok.convert_ids_to_tokens(input_protbert["input_ids"][1]))


print(60*"=")
###########################################################################
# ESM
###########################################################################
input_esm = esm2_8m_tok(SEQUENCES, return_tensors="pt", padding=True).to(DEVICE)
print("ESM2 tokenisation of the first sequence")
print("Raw seq :", SEQUENCES)
print("Input ESM:", input_esm)
print("Tokens first sequence :", esm2_8m_tok.convert_ids_to_tokens(input_esm["input_ids"][0]))
print("Tokens second sequence :", esm2_8m_tok.convert_ids_to_tokens(input_esm["input_ids"][1]))


## Get contextualised embeddings of the same dimensions

1. Pass inputs through the model. This automatically returns the last hidden state, which is an embedding of dimensions [batch size, sequence length, layer dimension (which is 1024 in ProtBert and 320 in ESM2 8M)].
2. To get embeddings of the same dimensions to use for downstream analysis, we need to perform a pooling operation. A typical procedure is averaging over the sequence length, to get a vector of length "layer dimension".

        We saw the model  adds [CLS] at the start and [EOS] at the end of every sequence. These are not amino acids - they're tokens the model uses internally.
        If we average all the embeddings together to get one vector per sequence, we don't want these artificial tokens pulling the average away from the actual biology.
        So after the model runs, we modify the mask to also zero out those positions.
        Then we average only what's left - the actual amino acid embeddings.
        We have a helper function to help us do this below:

In [ ]:

# -----------------------------------------------------------
# Helper: mean-pool while dropping special tokens
# -----------------------------------------------------------
def mean_pool_masked(embeddings: torch.Tensor,
                     attention_mask: torch.Tensor) -> torch.Tensor:
    """
    embeddings:     [batch, seq_len, hidden]
    attention_mask:   [batch, seq_len]  (1 = real token, 0 = pad)
    Returns:          [batch, hidden]
    """
    # Clone mask so we can zero-out CLS (position 0) and EOS (last real pos)
    mask = attention_mask.clone().float()
    lengths = attention_mask.sum(dim=1)          # includes specials & padding

    # Mask out <cls> / [CLS]
    mask[:, 0] = 0

    # Mask out <eos> / [SEP] (the last non-padded position for each sequence)
    batch_idx = torch.arange(mask.size(0), device=mask.device)
    mask[batch_idx, lengths - 1] = 0

    # Expand and pool
    mask_exp = mask.unsqueeze(-1)               # [batch, seq_len, 1]
    summed = (embeddings * mask_exp).sum(dim=1) # [batch, hidden]
    avg = summed / mask.sum(dim=1, keepdim=True).clamp(min=1)
    return avg



In [ ]:

###########################################################################
# PROTBERT
###########################################################################
# Input
input_protbert = protbert_tok(protbert_input_str, return_tensors="pt", padding=True).to(DEVICE)
# Pass to model
with torch.no_grad():
    protbert_out = protbert_model(**input_protbert)

# Mask out special tokens and do average pooling to get final contextualised embeddings.
protbert_embeddings = mean_pool_masked(
    protbert_out.last_hidden_state,
    input_protbert["attention_mask"]
).cpu().numpy()

print("ProtBERT embedding shape :", protbert_embeddings.shape)   # (2, 1024)


print(60*"=")
###########################################################################
# ESM
###########################################################################
# Input
input_esm = esm2_8m_tok(SEQUENCES, return_tensors="pt", padding=True).to(DEVICE)
# Pass to model
with torch.no_grad():
    esm_out = esm2_8m_model(**input_esm)

# Mask out special tokens and do average pooling to get final contextualised embeddings.
esm_embeddings = mean_pool_masked(
    esm_out.last_hidden_state,
    input_esm["attention_mask"]
).cpu().numpy()

print("ESM embedding shape :", esm_embeddings.shape)   # (2, 320)



Now we can use these embeddings for any downstream task!

# Part 2: TCR language-model workshop: CD4/CD8 classification

This workshop compares simple ways of representing TCR sequences for CD4/CD8 classification.

We compare:

- a simple raw-text baseline,
- SCEPTR embeddings, which are designed for TCRs,
- ESM-2 embeddings, from a general protein language model.

Most of the setup code is hidden/collapsed so that the visible notebook stays suitable for a beginner-friendly workshop.

In [ ]:
#@title Install packages { display-mode: "form" }
!pip install -q sceptr tidytcells scikit-learn umap-learn matplotlib pandas transformers torch


In [ ]:
#@title Hidden setup: load, clean, standardise and downsample the data { display-mode: "form" }

from pathlib import Path
import zipfile
import warnings
import urllib.request

import numpy as np
import pandas as pd
from IPython.display import display

import tidytcells as tt
import sceptr

RANDOM_STATE = 42  #@param {type:"integer"}
N_PER_CLASS = 1000 #@param {type:"integer"}

DATA_URL = "https://zenodo.org/records/13119615/files/Donor7_TCR_PBMC_final.csv.zip?download=1"
ZIP_PATH = Path("Donor7_TCR_PBMC_final.csv.zip")


def load_data(url=DATA_URL, zip_path=ZIP_PATH):
    """Download and read the 10x contig table."""
    if not zip_path.exists() or not zipfile.is_zipfile(zip_path):
        if zip_path.exists():
            zip_path.unlink()
        urllib.request.urlretrieve(url, zip_path)

    if not zipfile.is_zipfile(zip_path):
        raise RuntimeError("Download failed: the file is not a valid zip archive.")

    with zipfile.ZipFile(zip_path) as z:
        csv_files = [
            name for name in z.namelist()
            if name.endswith(".csv") and not name.startswith("__MACOSX/")
        ]
        if len(csv_files) != 1:
            raise RuntimeError(f"Expected one CSV file, found: {csv_files}")

        with z.open(csv_files[0]) as f:
            return pd.read_csv(f)


def standardise_v_gene(v_gene):
    """Convert a V gene name to the format expected by SCEPTR."""
    if pd.isna(v_gene) or str(v_gene).strip() == "":
        return np.nan

    try:
        out = tt.tr.standardize(
            str(v_gene).strip(),
            species="homosapiens",
            precision="allele",
            enforce_functional=True,
            log_failures=False,
        )
    except Exception:
        out = None

    return out if out is not None else np.nan


def standardise_j_gene(j_gene):
    """Add the common *01 allele if the J gene has no allele specified."""
    if pd.isna(j_gene) or str(j_gene).strip() == "":
        return None

    j_gene = str(j_gene).strip()
    return j_gene if "*" in j_gene else j_gene + "*01"


def standardise_cdr3(cdr3, j_gene=None):
    """Standardise a CDR3/junction sequence."""
    if pd.isna(cdr3) or str(cdr3).strip() == "":
        return np.nan

    try:
        out = tt.junction.standardize(
            str(cdr3).strip().upper(),
            j_symbol=standardise_j_gene(j_gene),
            species="homosapiens",
            on_fail="reject",
            log_failures=False,
        )
    except Exception:
        out = None

    return out if out is not None else np.nan


def make_paired_tcr_table(df):
    """Convert the 10x contig table into one row per cell with paired TCR information."""
    required = {
        "barcode", "chain", "v_gene", "j_gene", "cdr3",
        "productive", "high_confidence"
    }
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    x = df.copy()

    keep = (
        x["chain"].isin(["TRA", "TRB"])
        & x["productive"].astype(str).str.upper().eq("TRUE")
        & x["high_confidence"].astype(str).str.upper().eq("TRUE")
        & x["cdr3"].notna()
        & x["v_gene"].notna()
    )
    x = x.loc[keep].copy()

    x["v_gene"] = x["v_gene"].map(standardise_v_gene)
    x["cdr3"] = x.apply(lambda row: standardise_cdr3(row["cdr3"], row["j_gene"]), axis=1)
    x = x.dropna(subset=["v_gene", "cdr3"])

    sort_cols = ["barcode", "chain"] + [c for c in ["umis", "reads"] if c in x.columns]
    ascending = [True, True] + [False] * (len(sort_cols) - 2)

    x = (
        x.sort_values(sort_cols, ascending=ascending)
         .drop_duplicates(["barcode", "chain"], keep="first")
    )

    alpha = (
        x[x["chain"] == "TRA"]
        .set_index("barcode")[["v_gene", "cdr3"]]
        .rename(columns={"v_gene": "TRAV", "cdr3": "CDR3A"})
    )

    beta = (
        x[x["chain"] == "TRB"]
        .set_index("barcode")[["v_gene", "cdr3"]]
        .rename(columns={"v_gene": "TRBV", "cdr3": "CDR3B"})
    )

    label_cols = [
        c for c in [
            "donor", "origin", "cluster",
            "annotation_L1", "annotation_L2", "annotation_L3", "annotation_L4"
        ]
        if c in df.columns
    ]

    labels = df.drop_duplicates("barcode").set_index("barcode")[label_cols]

    return (
        alpha.join(beta, how="outer")
             .join(labels, how="left")
             .dropna(subset=["TRAV", "CDR3A", "TRBV", "CDR3B"])
             .reset_index()
    )


def downsample_cd4_cd8(tcrs, n_per_class=N_PER_CLASS, random_state=RANDOM_STATE):
    """Make a balanced CD4/CD8 subset."""
    cd4_cd8 = tcrs[tcrs["annotation_L1"].isin(["CD4", "CD8"])].copy()
    n = min(n_per_class, cd4_cd8["annotation_L1"].value_counts().min())

    return (
        cd4_cd8
        .groupby("annotation_L1", group_keys=False)
        .sample(n=n, random_state=random_state)
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )


df = load_data()
tcrs_all = make_paired_tcr_table(df)

# drop identical TCRs (clonal expansions)
# (This could also be done instead on the nucleotide level)
tcrs_all['bioidentity'] = tcrs_all[['TRAV', 'CDR3A', 'TRBV', 'CDR3B']].apply(lambda row: '-'.join(row), axis=1)
tcrs_unique = tcrs_all.drop_duplicates(subset=['bioidentity'])

tcrs_workshop = downsample_cd4_cd8(tcrs_unique)

y = tcrs_workshop["annotation_L1"].map({"CD4": 0, "CD8": 1}).to_numpy()

print(f"Ready: {len(tcrs_workshop)} paired TCRs")
print(tcrs_workshop["annotation_L1"].value_counts())


## 1. Look at the prepared data

The hidden setup cell has downloaded the data, standardised the TCR format, kept paired alpha-beta TCRs, and made a balanced CD4/CD8 subset.


In [ ]:
display(tcrs_workshop[["TRAV", "CDR3A", "TRBV", "CDR3B", "annotation_L1"]].head())
tcrs_workshop["annotation_L1"].value_counts()


## 2. Compute or load SCEPTR embeddings

SCEPTR is designed specifically for TCR sequences. Here we use it to turn each paired alpha-beta TCR into a vector of numbers.

The first run can take a while, so the embeddings are saved and reused.


In [ ]:
sceptr_input = tcrs_workshop[["TRAV", "CDR3A", "TRBV", "CDR3B"]]
embeddings = sceptr.calc_vector_representations(sceptr_input)

embedding_columns = [f"sceptr_{i}" for i in range(embeddings.shape[1])]
embedded_tcrs = pd.concat(
    [tcrs_workshop.reset_index(drop=True),
      pd.DataFrame(embeddings, columns=embedding_columns)],
    axis=1
  )

X_sceptr = embedded_tcrs.filter(like="sceptr_").to_numpy()

print("SCEPTR feature matrix:", X_sceptr.shape)

## 3. Compute or load ESM-2 embeddings

ESM-2 is a general protein language model. Unlike SCEPTR, it does not know that these are paired TCRs, so we embed the alpha and beta CDR3 amino-acid sequences separately and concatenate the two vectors.

This gives us an external protein-language-model comparison.


In [ ]:
#@title Hidden setup: ESM-2 embedding helper functions { display-mode: "form" }

import torch
from transformers import AutoTokenizer, AutoModel

ESM2_MODEL_NAME = "facebook/esm2_t6_8M_UR50D"  # small and workshop-friendly
ESM2_BATCH_SIZE = 32 #@param {type:"integer"}


def embed_sequences_with_esm2(sequences, model_name=ESM2_MODEL_NAME, batch_size=ESM2_BATCH_SIZE):
    """Embed a list of amino-acid sequences with ESM-2."""
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    all_embeddings = []

    for start in range(0, len(sequences), batch_size):
        batch = [str(seq) for seq in sequences[start:start + batch_size]]

        encoded = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
        ).to(device)

        with torch.no_grad():
            output = model(**encoded)
            pooled = mean_pool_masked(output.last_hidden_state, encoded["attention_mask"])

        all_embeddings.append(pooled.cpu().numpy())

    return np.vstack(all_embeddings)


def make_paired_esm2_embeddings(tcrs):
    """Embed alpha and beta CDR3s separately, then concatenate the embeddings."""
    alpha = embed_sequences_with_esm2(tcrs["CDR3A"].tolist())
    beta = embed_sequences_with_esm2(tcrs["CDR3B"].tolist())
    return np.concatenate([alpha, beta], axis=1)


In [ ]:
X_esm2 = make_paired_esm2_embeddings(tcrs_workshop)

print("ESM-2 feature matrix:", X_esm2.shape)

In [ ]:
#@title Hidden setup: modelling and plotting helper functions { display-mode: "form" }

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    ConfusionMatrixDisplay,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder

def categorical_logistic_model():
    """Logistic regression for categorical data."""
    return make_pipeline(
        OneHotEncoder(handle_unknown='ignore'),
        LogisticRegression(max_iter=1000, class_weight='balanced')
    )

def logistic_model():
    """Logistic regression for numerical embeddings."""
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight="balanced")
    )


def small_neural_network():
    """Small neural network for numerical embeddings."""
    return make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(64),
            max_iter=50,
            early_stopping=True,
            random_state=RANDOM_STATE
        )
    )

def select_rows(X, idx):
    """Select rows from either numpy arrays or pandas objects."""
    if hasattr(X, "iloc"):
        return X.iloc[idx]
    return X[idx]


def train_models(model_specs):
    """Train all models on the same train/test split and store predictions."""
    train_idx, test_idx, y_train, y_test = train_test_split(
        np.arange(len(y)),
        y,
        test_size=0.25,
        random_state=RANDOM_STATE,
        stratify=y
    )

    outputs = {}

    for name, X, model in model_specs:
        X_train = select_rows(X, train_idx)
        X_test = select_rows(X, test_idx)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_score = model.predict_proba(X_test)[:, 1]

        outputs[name] = {
            "model": model,
            "X_test": X_test,
            "y_test": y_test,
            "y_pred": y_pred,
            "y_score": y_score,
            "accuracy": accuracy_score(y_test, y_pred),
            "ROC AUC": roc_auc_score(y_test, y_score),
        }

    return outputs


def summarise_outputs(outputs):
    return pd.DataFrame({
        name: {
            "accuracy": out["accuracy"],
            "ROC AUC": out["ROC AUC"],
        }
        for name, out in outputs.items()
    }).T.sort_values("ROC AUC", ascending=False)


def plot_roc_curves(outputs):
    plt.figure(figsize=(6, 5))

    for name, out in outputs.items():
        RocCurveDisplay.from_predictions(
            out["y_test"],
            out["y_score"],
            name=name,
            ax=plt.gca()
        )

    plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
    plt.title("ROC curves")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.legend(fontsize=8)
    plt.show()


def plot_precision_recall_curves(outputs):
    plt.figure(figsize=(6, 5))

    for name, out in outputs.items():
        PrecisionRecallDisplay.from_predictions(
            out["y_test"],
            out["y_score"],
            name=name,
            ax=plt.gca()
        )

    plt.title("Precision-recall curves")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.legend(fontsize=8)
    plt.show()



## 4. Compare representations

Now we test whether CD4/CD8 classification works better using:

- raw TCR text,
- SCEPTR embeddings,
- ESM-2 CDR3 embeddings.

For each representation, we try a logistic regression classifier and a small neural network.

The raw-text model is included as a simple baseline. If it performs well, that is useful to show: a language-model embedding is not automatically better for every small supervised task.

In [ ]:
X_v_genes = tcrs_workshop[["TRAV", "TRBV"]]
model_specs = [
    ("SCEPTR + logistic regression", X_sceptr, logistic_model()),
    ("SCEPTR + neural network", X_sceptr, small_neural_network()),
    ("ESM-2 CDR3 + logistic regression", X_esm2, logistic_model()),
    ("ESM-2 CDR3 + neural network", X_esm2, small_neural_network()),
    ("V genes + logistic regression", X_v_genes, categorical_logistic_model()),
  ]

model_outputs = train_models(model_specs)
summary = summarise_outputs(model_outputs)
summary


## 5. ROC and precision-recall curves

Accuracy is useful, but ROC and precision-recall curves show more about how well each model separates the two classes across different thresholds.


In [ ]:
plot_roc_curves(model_outputs)
plot_precision_recall_curves(model_outputs)


### 6. Visualise the embedding spaces

UMAP compresses the embeddings into two dimensions. This is not used for classification here; it is only a visual aid.

If CD4 and CD8 separate more clearly in one embedding space than another, that can help explain why one representation performs better.


In [ ]:
import umap
import matplotlib.pyplot as plt


def plot_umap_embedding(X, title):
    xy = umap.UMAP(
        n_neighbors=30,
        min_dist=0.1,
        random_state=RANDOM_STATE
    ).fit_transform(X)

    plot_df = tcrs_workshop.copy()
    plot_df["UMAP1"] = xy[:, 0]
    plot_df["UMAP2"] = xy[:, 1]

    plt.figure(figsize=(6, 5))

    for label in ["CD4", "CD8"]:
        subset = plot_df[plot_df["annotation_L1"] == label]
        plt.scatter(subset["UMAP1"], subset["UMAP2"], s=5, alpha=0.6, label=label)

    plt.xlabel("UMAP1")
    plt.ylabel("UMAP2")
    plt.title(title)
    plt.legend()
    plt.show()


plot_umap_embedding(X_sceptr, "SCEPTR embedding space")
plot_umap_embedding(X_esm2, "ESM-2 CDR3 embedding space")


## Notes for workshop leads

The setup code is hidden/collapsed using notebook metadata and Colab form-style cells. In Google Colab, cells starting with `#@title ... { display-mode: "form" }` can be collapsed and shown as form-like setup cells.

The V-gene logistic regression baseline is intentionally included as a comparison. It helps participants see that a protein language model is a representation choice, not a guarantee of better performance. The main workshop discussion can then focus on when general protein embeddings, TCR-specific embeddings and simple text features may each be useful.

The ESM-2 model used here is the smallest ESM-2 model from Hugging Face, so it is more practical for workshops than larger protein language models. The embeddings are cached after the first run.